# XAI Sensitivity & Stability (L4 GPU)

**런타임: L4 GPU** (런타임 → 런타임 유형 변경 → L4)

### 준비 (한 번만)
Colab 업로드 위젯이 불안정하므로 **Google Drive**를 사용합니다.
1. https://drive.google.com 접속
2. 내 드라이브(MyDrive) 최상위에 아래 2개 파일을 드래그해서 업로드
   - `fireimage_clean.zip`  (약 32MB)
   - `weights_E.zip`  (약 590MB)
3. 업로드 끝나면 아래 셀을 순서대로 실행

In [ ]:
# 1) 클론 + 패키지 (shap 제거 → 설치 빠름)
%cd /content
!rm -rf fireimage_detection
!git clone https://github.com/yuntaewon812/fireimage_detection.git fireimage_detection -q
!pip install timm grad-cam lime scikit-image -q
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 2) Google Drive 마운트 + zip 복사
from google.colab import drive
import os, shutil
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
for fn in ['fireimage_clean.zip', 'weights_E.zip']:
    src = os.path.join(DRIVE, fn)
    if os.path.exists(src):
        shutil.copy(src, f'/content/{fn}')
        print(f'복사 완료: {fn} ({os.path.getsize(src)/1e6:.1f}MB)')
    else:
        print(f'없음: {src}  ← MyDrive 최상위에 올렸는지 확인')

In [ ]:
# 3) 압축 해제
import zipfile, os
BASE = '/content/fireimage_detection'

if os.path.exists('/content/fireimage_clean.zip'):
    !python {BASE}/colab_setup.py
    print('데이터 압축 해제 완료')
else:
    print('주의: fireimage_clean.zip 없음 — 셀 2 확인')

if os.path.exists('/content/weights_E.zip'):
    with zipfile.ZipFile('/content/weights_E.zip', 'r') as z:
        z.extractall(BASE)
    print('가중치 압축 해제 완료')
else:
    print('주의: weights_E.zip 없음 — 셀 2 확인')

w_dir = f'{BASE}/model_save/fireimage_abl_E/fold0'
print('fold0 가중치:', os.listdir(w_dir) if os.path.exists(w_dir) else '없음')

In [ ]:
# 4) Sensitivity & Stability 실행 (L4 GPU)
%cd /content/fireimage_detection
import time
t0 = time.time()
!python sensitivity_stability.py
gpu_total = time.time() - t0
print(f'\n총 GPU 실행 시간: {gpu_total:.1f}s ({gpu_total/60:.1f}분)')

In [ ]:
# 5) 결과 표시
import pandas as pd
gpu_df = pd.read_csv('results_SENS_STAB/sens_stab_cuda.csv')
print('=== GPU (L4) Sensitivity & Stability ===')
print(gpu_df[['Model','Method','Sensitivity','Stability','Time_sec']].to_string(index=False))

In [ ]:
# 6) 결과 다운로드
from google.colab import files
files.download('results_SENS_STAB/sens_stab_cuda.csv')